In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
collection_name="chat_messages"

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

# Initialize Qdrant client (in-memory)
client = QdrantClient(":memory:")

# Create collection (size depends on embedding model)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)  # 384 is common for "all-MiniLM-L6-v2"
)

In [ ]:
# Sample new message (in a real app, this would be dynamic)
import uuid

from qdrant_client.models import PointStruct

new_message = "What's the weather in Paris today?"

# Generate the embedding for the new message
new_message_embedding = model.encode([new_message]).tolist()

# Add this message to Qdrant collection
point = PointStruct(
    id=str(uuid.uuid4()),  # can be string or int
    vector=new_message_embedding[0],
    payload={"text": new_message}
)

client.upsert(
    collection_name=collection_name, 
    points=[point]
)

In [ ]:
# User query (dynamic input)
user_query = "Is it sunny in Paris today?"

# Generate the embedding for the query
query_embedding = model.encode([user_query]).tolist()

# Perform search for similar messages in Qdrant
results = client.query_points(
    collection_name=collection_name,
    query=query_embedding[0],
    limit=1
)

current_id = ""

# Display the search results
for point in results.points:
    print(f"{point.model_dump_json(indent=4)}")
    current_id = point.id

In [ ]:
from qdrant_client.models import PointIdsList

client.delete(
    collection_name=collection_name,
    points_selector=PointIdsList(points=[current_id])
)